In [3]:
import pandas as pd
import numpy as np
from pathlib import Path


In [77]:
current = Path.cwd()
project_root = current if (current / "CLAUDE.md").exists() else current.parent.parent

precipitation_review = project_root / "data" / "raw" / "precipitation" / "precipitation_review.csv"
crop_yield_review = project_root / "data" / "raw" / "crop_yield" / "crop_yield_review.csv"
vegetation_review = project_root / "data" / "raw" / "vegetation" / "vegetation_review.csv"
geography_common = project_root / "data" / "raw" / "geography" / "geography_common.csv"


In [24]:
current = Path.cwd()
project_root = current if (current / "CLAUDE.md").exists() else current.parent.parent

pickle_folder_path = project_root / "data" / "raw" / "pickle"

crop_yield_df = pd.read_pickle(pickle_folder_path/"df_crop_yield_2016_2026.pkl")
geography_df = pd.read_pickle(pickle_folder_path/"df_geography.pkl")
precipitation_df = pd.read_pickle(pickle_folder_path/"df_precipitation_2016_2026.pkl")
vegetation_df = pd.read_pickle(pickle_folder_path/"df_vegetation_2016_2026.pkl")

In [25]:
# renaming the columns

crop_yield_df.rename(
    columns={"state_name": "state", "district_name": "district"}, inplace=True
)
geography_df.rename(columns={"State": "state", "District": "district"}, inplace=True)
precipitation_df.rename(
    columns={"State_Name": "state", "District_Name": "district"}, inplace=True
)
vegetation_df.rename(
    columns={"state_name": "state", "district_name": "district"}, inplace=True
)

In [28]:
# Determining the common state names and removing unwanted records

states_cy = set(crop_yield_df["state"].unique())
states_geo = set(geography_df["state"].unique())
states_per = set(precipitation_df["state"].unique())
states_veg = set(vegetation_df["state"].unique())

common_states = states_cy & states_geo & states_per & states_veg

common_states_df = pd.DataFrame(list(common_states), columns=["state"])
crop_yield_df = crop_yield_df[crop_yield_df["state"].isin(common_states_df["state"])]
geography_df = geography_df[geography_df["state"].isin(common_states_df["state"])]
precipitation_df = precipitation_df[
    precipitation_df["state"].isin(common_states_df["state"])
]
vegetation_df = vegetation_df[vegetation_df["state"].isin(common_states_df["state"])]

In [36]:
# Cleaning up the districts data. Take geography as the base for the canonical dataset

# crop_yield_df['district'].nunique() 608
# geography_df['district'].nunique() 511
# precipitation_df['district'].nunique() 587
# vegetation_df['district'].nunique() 511



In [57]:

from rapidfuzz import process

common_districts_df = geography_df[['state', 'district']].drop_duplicates()

state_districts = common_districts_df.groupby('state')['district'].apply(list).to_dict()

def district_match(row):
    state = row['state']
    district = row['district']

    districts_by_state = state_districts.get(state,[])
    match, score, _ = process.extractOne(district, districts_by_state)
    return pd.Series([match, score])

crop_yield_df[['mapped_district', 'score']] =  crop_yield_df.apply(district_match, axis =1)
precipitation_df[['mapped_district', 'score']] =  precipitation_df.apply(district_match, axis =1)
vegetation_df[['mapped_district', 'score']] =  vegetation_df.apply(district_match, axis =1)





In [81]:
common_districts_df.to_csv(geography_common, index=False)

In [76]:
crop_yield_review_df = crop_yield_df[crop_yield_df['score'] < 95][['district','mapped_district']].drop_duplicates()
crop_yield_review_df.to_csv(crop_yield_review, index=False)

In [75]:
precipitation_review_df = precipitation_df[precipitation_df['score'] < 95][['district','mapped_district']].drop_duplicates()
precipitation_review_df.to_csv(precipitation_review, index=False)

In [74]:
vegetation_review_df = vegetation_df[vegetation_df['score'] < 95][['district','mapped_district']].drop_duplicates()
vegetation_review_df.to_csv(vegetation_review, index=False)